# Visionlytics - Train Extreme Density Model (CSRNet) 🧠

Run this notebook on Google Colab (with a **T4 GPU** enabled) to train the CSRNet density regression model on the ShanghaiTech dataset. Follow the step-by-step instructions carefully!

### Step 1: Clone the Repository & Install Dependencies

In [ ]:
!git clone https://github.com/ivin-baiju/Visionlytics.git
%cd Visionlytics
!pip install -e backend/
!pip install scipy h5py

### Step 2: Download ShanghaiTech Crowd Dataset (~4GB)
This will download the official dataset used for crowd counting algorithms.

In [ ]:
import os
!wget -q --show-progress "https://www.dropbox.com/scl/fi/dkj5kulc9zj0rzesslck8/ShanghaiTech_Crowd_Counting_Dataset.zip?rlkey=ymbcj50ac04uvqn8p49j9af5f&dl=1" -O ShanghaiTech.zip
!unzip -q ShanghaiTech.zip -d backend/dataset/
# Ensure folder structure
!mkdir -p backend/dataset/ShanghaiTech
!mv backend/dataset/part_A backend/dataset/ShanghaiTech/ 2>/dev/null || true
!mv backend/dataset/part_B backend/dataset/ShanghaiTech/ 2>/dev/null || true


### Step 3: Define the Dataset Loader and Density Map Generator
This converts the .mat ground truth files (head coordinates) into visual heatmap tensors for PyTorch.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import cv2
import numpy as np
import scipy.io as io
import glob

class ShanghaiTechDataset(Dataset):
    def __init__(self, root_path, transform=None):
        self.img_paths = glob.glob(os.path.join(root_path, 'images', '*.jpg'))
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, index):
        img_path = self.img_paths[index]
        gt_path = img_path.replace('.jpg', '.mat').replace('images', 'ground_truth').replace('IMG_', 'GT_IMG_')
        
        # Load Image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Load Ground Truth (Head points)
        mat = io.loadmat(gt_path)
        pts = mat['image_info'][0,0][0,0][0]
        
        # Create Density Map (Simplified 1/8 scale for CSRNet output)
        h, w = img.shape[:2]
        density_map = np.zeros((h // 8, w // 8), dtype=np.float32)
        
        for pt in pts:
            x, y = int(pt[0] / 8), int(pt[1] / 8)
            if 0 <= y < density_map.shape[0] and 0 <= x < density_map.shape[1]:
                density_map[y, x] = 1.0 # Simplified point map (No gaussian blur for speed in this demo)
                
        if self.transform:
            img = self.transform(img)
            
        return img, torch.from_numpy(density_map).unsqueeze(0)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ShanghaiTechDataset('backend/dataset/ShanghaiTech/part_B/train_data', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True) # Batch size 1 because images have different sizes


### Step 4: Run the Training Loop
This will train the model for 10 epochs. It will take a few hours depending on the GPU.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from backend.computer_vision.csrnet import CSRNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

model = CSRNet(load_weights=False).to(device)
criterion = nn.MSELoss(size_average=False).to(device)
optimizer = optim.SGD(model.parameters(), lr=1e-7, momentum=0.95, weight_decay=5 * 1e-4)

epochs = 10
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    
    for i, (img, target) in enumerate(train_loader):
        img = img.to(device)
        target = target.to(device)
        
        optimizer.zero_grad()
        output = model(img)
        
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        if i % 100 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i}/{len(train_loader)}], Loss: {loss.item():.4f}")
            
    print(f"Epoch {epoch+1} Completed! Average Loss: {epoch_loss/len(train_loader):.4f}")
    
    # Save checkpoint after every epoch
    torch.save(model.state_dict(), f"csrnet_epoch_{epoch+1}.pth")

print("Training Complete! Download the .pth files from Colab!")